# Marker2 Annotation Comparison Supplementary Figure

This notebook builds the comparative annotation supplementary figure and its source data from Nextflow workflow outputs.

Expected workflow inputs:

- `FULL_FEATURE_RUN_OUTDIR`: comparison run using the full annotation feature sets.
- `MATCHED_FEATURE_RUN_OUTDIR`: comparison run using the filtered/matched CAT feature set.
- `DIVERGENCE_RUN_OUTDIR`: run used for the GRCh38 divergence and CDS-reference panels. By default this is the matched run.

Outputs:

- panel-level source data under `OUTPUT_ROOT/source_data/`
- two supplementary figure layouts under `OUTPUT_ROOT/figures/`
- `source_data_manifest.tsv` linking panels to source-data files

The notebook does not depend on `supplementary_tables.ipynb`. The supplementary tables should be generated separately.


In [ ]:
from pathlib import Path
import shutil
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Find the hprc-qc checkout whether the notebook is run from repo root or notebooks/."""
    for path in [start, *start.parents]:
        if (path / 'nextflow' / 'pipelines' / 'ensembl_cat_comparison').exists():
            return path
    raise RuntimeError(f'Could not find hprc-qc repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())

# Change these paths when rerunning on the HPC.
# The local defaults allow the notebook to execute against the checked-in draft data cache.
FULL_FEATURE_RUN_OUTDIR = REPO_ROOT / 'data' / 'with_likely_coding'
MATCHED_FEATURE_RUN_OUTDIR = REPO_ROOT / 'data' / 'with_likely_coding'
DIVERGENCE_RUN_OUTDIR = MATCHED_FEATURE_RUN_OUTDIR

# Local draft fallback only. Set to False for the final HPC rerun if you want missing
# divergence/CDS workflow outputs to fail loudly rather than use cached source tables.
ALLOW_PRECOMPUTED_FALLBACK = True
PRECOMPUTED_SUPP_DIR = REPO_ROOT / 'data' / 'HPRC_Annotation_Supp'

OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'marker2_annotation_comparison_supplementary_figure'
SOURCE_DATA_DIR = OUTPUT_ROOT / 'source_data'
FIGURE_DIR = OUTPUT_ROOT / 'figures'

PANEL_A_DIR = SOURCE_DATA_DIR / 'panel_A_transcript_concordance'
PANEL_B_DIR = SOURCE_DATA_DIR / 'panel_B_divergence_by_biotype'
PANEL_C_DIR = SOURCE_DATA_DIR / 'panel_C_cds_reference_comparison'

for path in [PANEL_A_DIR, PANEL_B_DIR, PANEL_C_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Full feature run:    {FULL_FEATURE_RUN_OUTDIR}')
print(f'Matched feature run: {MATCHED_FEATURE_RUN_OUTDIR}')
print(f'Divergence/CDS run:  {DIVERGENCE_RUN_OUTDIR}')
print(f'Source data:         {SOURCE_DATA_DIR}')
print(f'Figures:             {FIGURE_DIR}')


In [ ]:
def require_file(path: Path, label: str) -> Path:
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing required {label}: {path}')
    return path


def copy_required(src: Path, dst: Path, label: str):
    src = require_file(src, label)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return dst


def read_required_tsv(path: Path, label: str) -> pd.DataFrame:
    path = require_file(path, label)
    return pd.read_csv(path, sep='	')


def write_tsv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='	', index=False)
    return path


source_provenance = []


def note_source(panel: str, output: Path, input_path: Path, mode: str):
    source_provenance.append({
        'panel': panel,
        'output_file': str(output.relative_to(OUTPUT_ROOT)),
        'input_file': str(input_path),
        'source_mode': mode,
    })


# Panel A: transcript concordance from two explicit workflow runs.
full_intron_dir = FULL_FEATURE_RUN_OUTDIR / 'intermediate_spreadsheets' / 'intron_chain'
matched_intron_dir = MATCHED_FEATURE_RUN_OUTDIR / 'intermediate_spreadsheets' / 'intron_chain'

panel_a_inputs = [
    (
        'full_feature_set',
        full_intron_dir / 'intron_chain_full_denom_per_assembly.tsv',
        PANEL_A_DIR / 'intron_chain_full_denom_per_assembly.tsv',
        'Panel A full feature-set intron-chain table',
    ),
    (
        'matched_feature_set',
        matched_intron_dir / 'intron_chain_by_biotype_per_assembly.tsv',
        PANEL_A_DIR / 'intron_chain_by_biotype_per_assembly.tsv',
        'Panel A matched feature-set intron-chain table',
    ),
]

panel_a_frames = []
for comparison_set, src, dst, label in panel_a_inputs:
    df = read_required_tsv(src, label)
    df.insert(0, 'comparison_set', comparison_set)
    panel_a_frames.append(df)
    write_tsv(df, dst)
    note_source('A', dst, src, 'workflow_summary')

panel_a_source = pd.concat(panel_a_frames, ignore_index=True)
panel_a_combined = write_tsv(panel_a_source, PANEL_A_DIR / 'panel_A_transcript_concordance_source.tsv')
note_source('A', panel_a_combined, Path('<combined from full and matched workflow summaries>'), 'workflow_summary_combined')


DIV_CATEGORY_LABELS = {
    'both_agree_reference': ('Both agree (same as GENCODE v47)', 1),
    'both_agree_diverged': ('Both agree (diverged from GENCODE v47)', 2),
    'ensembl_specific_divergence': ('Ensembl-specific divergence', 3),
    'cat_specific_divergence': ('CAT-specific divergence', 4),
    'insufficient_data': ('Insufficient data', 5),
}
BIOTYPE_ORDER = {
    'protein_coding': 1,
    'processed_pseudogene': 2,
    'lncRNA': 3,
    'miRNA': 4,
    'snRNA': 5,
    'misc_RNA': 6,
}


def build_panel_b_from_workflow(src: Path) -> pd.DataFrame:
    df = read_required_tsv(src, 'Panel B workflow divergence-by-biotype table')
    expected = {'biotype', 'divergence_category', 'count'}
    if not expected.issubset(df.columns):
        raise ValueError(f'{src} does not contain expected columns: {sorted(expected)}')
    df = df[df['biotype'].isin(BIOTYPE_ORDER)].copy()
    df['ensembl_biotype'] = df['biotype']
    df['category_label'] = df['divergence_category'].map(lambda x: DIV_CATEGORY_LABELS.get(x, (x, 99))[0])
    df['category_order'] = df['divergence_category'].map(lambda x: DIV_CATEGORY_LABELS.get(x, (x, 99))[1])
    df['n_gene_pair_observations'] = df['count'].astype(float)
    totals = df.groupby('ensembl_biotype')['n_gene_pair_observations'].transform('sum')
    df['n_total_gene_pair_observations'] = totals
    df['pct_within_biotype'] = (df['n_gene_pair_observations'] / totals * 100).round(4)
    df['biotype_order'] = df['ensembl_biotype'].map(BIOTYPE_ORDER)
    cols = [
        'ensembl_biotype', 'divergence_category', 'category_label', 'category_order',
        'n_gene_pair_observations', 'n_total_gene_pair_observations',
        'pct_within_biotype', 'biotype_order',
    ]
    return df[cols].sort_values(['biotype_order', 'category_order'])


# Panel B: divergence categories within each biotype.
panel_b_workflow = DIVERGENCE_RUN_OUTDIR / 'intermediate_spreadsheets' / 'divergence' / 'grch38_divergence_by_biotype.tsv'
panel_b_out = PANEL_B_DIR / 'divergence_by_biotype_panel_B_source.tsv'
if panel_b_workflow.exists() and panel_b_workflow.stat().st_size > 0:
    panel_b_source = build_panel_b_from_workflow(panel_b_workflow)
    write_tsv(panel_b_source, panel_b_out)
    note_source('B', panel_b_out, panel_b_workflow, 'workflow_summary')
elif ALLOW_PRECOMPUTED_FALLBACK:
    fallback = PRECOMPUTED_SUPP_DIR / 'supp_divergence_by_biotype' / 'divergence_by_biotype_panel_B_source.tsv'
    copy_required(fallback, panel_b_out, 'Panel B precomputed source table')
    note_source('B', panel_b_out, fallback, 'precomputed_local_fallback')
    print(f'WARNING: Panel B used precomputed fallback: {fallback}')
else:
    require_file(panel_b_workflow, 'Panel B workflow divergence-by-biotype table')

definitions = PRECOMPUTED_SUPP_DIR / 'supp_divergence_by_biotype' / 'divergence_category_definitions.tsv'
if definitions.exists():
    copied = copy_required(definitions, PANEL_B_DIR / 'divergence_category_definitions.tsv', 'Panel B category definitions')
    note_source('B', copied, definitions, 'definitions')


CDS_LABELS = {
    'exact_match': 'Exact match',
    'in_frame_shorter': 'In-frame shorter',
    'in_frame_longer': 'In-frame longer',
    'frameshift_shorter': 'Frameshift shorter',
    'frameshift_longer': 'Frameshift longer',
    'coding_lost': 'Coding lost',
    'coding_gained': 'Coding gained',
    'non_coding': 'Non-coding',
}


def build_panel_c_from_workflow(src: Path):
    df = read_required_tsv(src, 'Panel C workflow CDS change-type cross-tab')
    expected = {'biotype', 'ens_cds_change_type', 'cat_cds_change_type', 'count'}
    if not expected.issubset(df.columns):
        raise ValueError(f'{src} does not contain expected columns: {sorted(expected)}')
    df = df[df['biotype'] == 'protein_coding'].copy()
    df['ensembl_plot_group'] = df['ens_cds_change_type'].map(CDS_LABELS).fillna(df['ens_cds_change_type'])
    df['cat_plot_group'] = df['cat_cds_change_type'].map(CDS_LABELS).fillna(df['cat_cds_change_type'])
    raw = df.rename(columns={
        'ens_cds_change_type': 'ensembl_cds_change_type',
        'cat_cds_change_type': 'cat_cds_change_type',
        'count': 'n_gene_pair_observations',
    })
    grouped = (
        raw.groupby(['ensembl_plot_group', 'cat_plot_group'], as_index=False)['n_gene_pair_observations']
        .sum()
        .sort_values(['ensembl_plot_group', 'cat_plot_group'])
    )
    return raw, grouped


# Panel C: CDS reference change-type flow for the protein-coding subset.
panel_c_workflow = DIVERGENCE_RUN_OUTDIR / 'intermediate_spreadsheets' / 'divergence' / 'grch38_cds_change_type_cross_tab.tsv'
panel_c_raw_out = PANEL_C_DIR / 'cds_reference_change_type_flow_counts.tsv'
panel_c_grouped_out = PANEL_C_DIR / 'cds_reference_change_type_flow_grouped_counts.tsv'
if panel_c_workflow.exists() and panel_c_workflow.stat().st_size > 0:
    panel_c_raw, panel_c_grouped = build_panel_c_from_workflow(panel_c_workflow)
    write_tsv(panel_c_raw, panel_c_raw_out)
    write_tsv(panel_c_grouped, panel_c_grouped_out)
    note_source('C', panel_c_raw_out, panel_c_workflow, 'workflow_summary')
    note_source('C', panel_c_grouped_out, panel_c_workflow, 'workflow_summary')
elif ALLOW_PRECOMPUTED_FALLBACK:
    raw_fallback = PRECOMPUTED_SUPP_DIR / 'supp_cds_reference_comparison_with_inset' / 'cds_reference_change_type_flow_counts.tsv'
    grouped_fallback = PRECOMPUTED_SUPP_DIR / 'supp_cds_reference_comparison_with_inset' / 'cds_reference_change_type_flow_grouped_counts.tsv'
    copy_required(raw_fallback, panel_c_raw_out, 'Panel C precomputed raw flow table')
    copy_required(grouped_fallback, panel_c_grouped_out, 'Panel C precomputed grouped flow table')
    note_source('C', panel_c_raw_out, raw_fallback, 'precomputed_local_fallback')
    note_source('C', panel_c_grouped_out, grouped_fallback, 'precomputed_local_fallback')
    print(f'WARNING: Panel C used precomputed fallback: {raw_fallback.parent}')
else:
    require_file(panel_c_workflow, 'Panel C workflow CDS change-type cross-tab')


provenance_path = write_tsv(pd.DataFrame(source_provenance), SOURCE_DATA_DIR / 'source_data_provenance.tsv')
print('Panel source data written:')
for path in sorted(SOURCE_DATA_DIR.rglob('*.tsv')):
    print(' -', path.relative_to(OUTPUT_ROOT))


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from matplotlib.path import Path as MplPath
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import FuncFormatter

COLORS = {
    "Both agree (same as ref)": "#2ecc71",
    "Both agree (diverged from ref)": "#3498db",
    "Ensembl-specific divergence": "#f39c12",
    "CAT-specific divergence": "#9b59b6",
    "Exact match": "#2ecc71",
    "Same intron chain": "#2c7fb8",
    "Partial overlap": "#f39c12",
    "No match": "#e74c3c",
    "Gene not shared": "#7f7f7f",
    "Coding gained": "#8c8c8c",
    "Coding lost": "#f39c12",
    "Frameshift longer": "#8e3b2f",
    "Frameshift shorter": "#e74c3c",
    "In-frame longer": "#2c7fb8",
    "In-frame shorter": "#5dade2",
    "Non-coding": "#bdbdbd",
}

DIV_LABELS = {
    "Both agree (same as ref)": "Both agree (same as GENCODE v47)",
    "Both agree (diverged from ref)": "Both agree (diverged from GENCODE v47)",
    "Ensembl-specific divergence": "Ensembl-specific divergence",
    "CAT-specific divergence": "CAT-specific divergence",
}

BIOTYPE_LABELS = {
    "protein_coding": "Protein-coding",
    "lncRNA": "lncRNA",
    "pseudogene": "Pseudogene",
    "processed_pseudogene": "Processed pseudogene",
    "other_ncRNA": "Other ncRNA",
    "other": "Other coding types",
}


def panel_letter(fig, x, y, letter):
    fig.text(x, y, letter, fontsize=13, fontweight="bold", ha="left", va="top")


def fmt_pct(x):
    if x >= 10:
        return f"{x:.0f}%"
    return f"{x:.1f}%"


def draw_stacked_barh(ax, y, values, labels, height=0.34, x0=0, show_labels=True):
    left = x0
    for lab, val in zip(labels, values):
        if val <= 0:
            continue
        ax.barh(y, val, left=left, height=height, color=COLORS[lab], edgecolor="white", linewidth=0.45)
        if show_labels and val >= 7:
            ax.text(left + val / 2, y, fmt_pct(val), ha="center", va="center", fontsize=7, color="white", fontweight="bold")
        left += val


def draw_panel_a(ax):
    df = pd.read_csv(PANEL_B_DIR / "divergence_by_biotype_panel_B_source.tsv", sep="\t")
    biotypes = df.sort_values("biotype_order")["ensembl_biotype"].drop_duplicates().tolist()
    cats = df.sort_values("category_order")["category_label"].drop_duplicates().tolist()
    x = np.arange(len(biotypes))
    width = 0.16
    offsets = (np.arange(len(cats)) - (len(cats) - 1) / 2) * width
    for cat in cats:
        vals = []
        for bt in biotypes:
            sub = df[(df["ensembl_biotype"] == bt) & (df["category_label"] == cat)]
            vals.append(float(sub["pct_within_biotype"].iloc[0]) if len(sub) else 0)
        ax.bar(
            x + offsets[cats.index(cat)],
            vals,
            width=width * 0.92,
            color=COLORS[cat],
            edgecolor="white",
            linewidth=0.45,
            label=DIV_LABELS.get(cat, cat),
        )
    ax.set_ylabel("Matched gene-pair\nobservations within\nbiotype (%)", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([BIOTYPE_LABELS.get(b, b) for b in biotypes], rotation=24, ha="right", fontsize=7)
    ax.tick_params(axis="y", labelsize=8)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", color="#dddddd", linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(
        loc="lower right",
        bbox_to_anchor=(0.985, 1.005),
        ncol=1,
        frameon=True,
        facecolor="white",
        edgecolor="#dddddd",
        framealpha=0.94,
        fontsize=6.6,
        handlelength=1.1,
        borderpad=0.35,
        labelspacing=0.35,
    )


def ribbon(ax, x0, y0a, y0b, x1, y1a, y1b, color, alpha=0.34):
    verts = [
        (x0, y0a),
        (x0 + 0.35 * (x1 - x0), y0a),
        (x1 - 0.35 * (x1 - x0), y1a),
        (x1, y1a),
        (x1, y1b),
        (x1 - 0.35 * (x1 - x0), y1b),
        (x0 + 0.35 * (x1 - x0), y0b),
        (x0, y0b),
        (x0, y0a),
    ]
    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]
    ax.add_patch(patches.PathPatch(MplPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha))


def draw_alluvial(ax, flow, scale_total=None, drop_exact_exact=False, min_flow_pct=0.0, label_side_pct_total=True):
    if drop_exact_exact:
        flow = flow[~((flow["ensembl_plot_group"] == "Exact match") & (flow["cat_plot_group"] == "Exact match"))].copy()
    groups = ["Exact match", "In-frame shorter", "In-frame longer", "Frameshift shorter", "Frameshift longer", "Coding lost", "Coding gained", "Non-coding"]
    groups = [g for g in groups if (flow["ensembl_plot_group"].eq(g).any() or flow["cat_plot_group"].eq(g).any())]
    flow = flow[flow["n_gene_pair_observations"] > 0].copy()
    if scale_total is None:
        scale_total = flow["n_gene_pair_observations"].sum()
    flow["pct"] = flow["n_gene_pair_observations"] / scale_total * 100.0
    flow = flow[flow["pct"] >= min_flow_pct]

    left_tot = flow.groupby("ensembl_plot_group")["pct"].sum().reindex(groups, fill_value=0)
    right_tot = flow.groupby("cat_plot_group")["pct"].sum().reindex(groups, fill_value=0)
    gap = 0.035
    total_height = max(left_tot.sum(), right_tot.sum())
    if total_height == 0:
        return
    norm = 1.0 / (total_height + gap * (len(groups) - 1))
    left_pos, right_pos = {}, {}
    y = 1.0
    for g in groups:
        h = left_tot[g] * norm
        if h > 0:
            left_pos[g] = [y - h, y]
            y -= h + gap
    y = 1.0
    for g in groups:
        h = right_tot[g] * norm
        if h > 0:
            right_pos[g] = [y - h, y]
            y -= h + gap

    left_cursor = {g: left_pos[g][0] for g in left_pos}
    right_cursor = {g: right_pos[g][0] for g in right_pos}
    for _, row in flow.sort_values("pct", ascending=False).iterrows():
        lg, rg, pct = row["ensembl_plot_group"], row["cat_plot_group"], row["pct"]
        if lg not in left_cursor or rg not in right_cursor:
            continue
        h = pct * norm
        y0a, y0b = left_cursor[lg], left_cursor[lg] + h
        y1a, y1b = right_cursor[rg], right_cursor[rg] + h
        ribbon(ax, 0.18, y0a, y0b, 0.82, y1a, y1b, COLORS.get(lg, "#999999"))
        left_cursor[lg] += h
        right_cursor[rg] += h

    for side, positions, totals, x, ha, text_x in [
        ("Ensembl", left_pos, left_tot, 0.16, "right", 0.13),
        ("CAT", right_pos, right_tot, 0.84, "left", 0.87),
    ]:
        for g, (lo, hi) in positions.items():
            ax.add_patch(patches.Rectangle((x - 0.02, lo), 0.04, hi - lo, color=COLORS.get(g, "#999999"), ec="white", lw=0.4))
            pct = totals[g]
            if pct >= 0.25 or not label_side_pct_total:
                ax.text(
                    text_x,
                    (lo + hi) / 2,
                    f"{pct:.1f}%",
                    ha=ha,
                    va="center",
                    fontsize=8,
                    color="#444444",
                    bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.8, "alpha": 0.85},
                )
        ax.text(x, -0.075, side, ha="center", va="top", fontsize=8)
    ax.add_patch(
        patches.Rectangle((0.02, 0.02), 0.96, 0.96, transform=ax.transAxes, fill=False, ec="#666666", lw=0.6)
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.13, 1.03)
    ax.axis("off")


def draw_cds_alluvial(
    ax,
    rows,
    order,
    color_map,
    percent_denominator=None,
    show_labels=True,
    label_filter=None,
    label_fmt="{pct:.1f}%",
    min_label_pct=0.0,
    label_bbox=True,
    fontsize=8,
    x_ens=0.18,
    x_cat=0.82,
    bar_w=0.055,
    label_offset=0.055,
    ribbon_alpha=0.30,
):
    pivot = rows.pivot_table(
        index="ensembl_plot_group",
        columns="cat_plot_group",
        values="n_gene_pair_observations",
        fill_value=0,
    )
    pivot = pivot.reindex(index=order, columns=order, fill_value=0)
    grand_total = float(pivot.values.sum())
    if grand_total == 0:
        return {}, {}, 0
    if percent_denominator is None:
        percent_denominator = grand_total

    ens_totals = {lab: float(pivot.loc[lab].sum()) for lab in order}
    cat_totals = {lab: float(pivot[lab].sum()) for lab in order}

    def cumulative_ranges(totals):
        ranges = {}
        cursor = 0.0
        for lab in order:
            frac = totals[lab] / grand_total
            ranges[lab] = (cursor, cursor + frac)
            cursor += frac
        return ranges

    ens_ranges = cumulative_ranges(ens_totals)
    cat_ranges = cumulative_ranges(cat_totals)

    for lab, (lo, hi) in ens_ranges.items():
        if hi > lo:
            ax.bar(x_ens, hi - lo, width=bar_w, bottom=lo, color=color_map.get(lab, "#999999"), align="center", zorder=2)
    for lab, (lo, hi) in cat_ranges.items():
        if hi > lo:
            ax.bar(x_cat, hi - lo, width=bar_w, bottom=lo, color=color_map.get(lab, "#999999"), align="center", zorder=2)

    ens_cursor = {k: v[0] for k, v in ens_ranges.items()}
    cat_cursor = {k: v[0] for k, v in cat_ranges.items()}
    for e in order:
        for c in order:
            cnt = float(pivot.loc[e, c])
            if cnt <= 0:
                continue
            frac = cnt / grand_total
            e_lo, e_hi = ens_cursor[e], ens_cursor[e] + frac
            c_lo, c_hi = cat_cursor[c], cat_cursor[c] + frac
            ens_cursor[e] = e_hi
            cat_cursor[c] = c_hi
            cx = (x_ens + x_cat) / 2.0
            verts = [
                (x_ens + bar_w / 2, e_lo),
                (cx, e_lo),
                (cx, c_lo),
                (x_cat - bar_w / 2, c_lo),
                (x_cat - bar_w / 2, c_hi),
                (cx, c_hi),
                (cx, e_hi),
                (x_ens + bar_w / 2, e_hi),
                (x_ens + bar_w / 2, e_lo),
            ]
            codes = [
                MplPath.MOVETO,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.LINETO,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.CURVE4,
                MplPath.CLOSEPOLY,
            ]
            ax.add_patch(
                patches.PathPatch(
                    MplPath(verts, codes),
                    facecolor=color_map.get(e, "#999999"),
                    edgecolor="none",
                    alpha=ribbon_alpha,
                    zorder=1,
                )
            )

    if show_labels:
        bbox = {"facecolor": "white", "edgecolor": "none", "pad": 0.8, "alpha": 0.82} if label_bbox else None
        label_entries = []
        for side, ranges, totals, x, ha, dx in [
            ("ensembl", ens_ranges, ens_totals, x_ens, "right", -label_offset),
            ("cat", cat_ranges, cat_totals, x_cat, "left", label_offset),
        ]:
            for lab, (lo, hi) in ranges.items():
                cnt = totals[lab]
                if cnt <= 0:
                    continue
                pct = cnt / percent_denominator * 100.0
                if pct < min_label_pct:
                    continue
                if label_filter and not label_filter(lab, side):
                    continue
                label_entries.append(
                    {
                        "side": side,
                        "x": x + dx,
                        "y": (lo + hi) / 2,
                        "ha": ha,
                        "text": label_fmt.format(lab=lab, cnt=int(round(cnt)), pct=pct),
                    }
                )

        for side in ("ensembl", "cat"):
            entries = [entry for entry in label_entries if entry["side"] == side]
            entries.sort(key=lambda entry: entry["y"])
            if len(entries) > 1:
                min_gap = 0.070 if fontsize <= 8 else 0.055
                lo_bound, hi_bound = 0.030, 0.970
                for i in range(1, len(entries)):
                    entries[i]["y"] = max(entries[i]["y"], entries[i - 1]["y"] + min_gap)
                overflow = entries[-1]["y"] - hi_bound
                if overflow > 0:
                    for entry in entries:
                        entry["y"] -= overflow
                if entries[0]["y"] < lo_bound:
                    shift = lo_bound - entries[0]["y"]
                    for entry in entries:
                        entry["y"] += shift
                for i in range(1, len(entries)):
                    entries[i]["y"] = max(entries[i]["y"], entries[i - 1]["y"] + min_gap)
                overflow = entries[-1]["y"] - hi_bound
                if overflow > 0:
                    for entry in entries:
                        entry["y"] -= overflow
            for entry in entries:
                ax.text(
                    entry["x"],
                    entry["y"],
                    entry["text"],
                    ha=entry["ha"],
                    va="center",
                    fontsize=fontsize,
                    color="#333333",
                    bbox=bbox,
                    clip_on=False,
                    zorder=3,
                )

    ax.set_xlim(-0.08, 1.08)
    ax.set_ylim(0, 1)
    ax.set_xticks([x_ens, x_cat])
    ax.set_xticklabels(["Ensembl", "CAT"], fontsize=8)
    ax.set_yticks([])
    return ens_ranges, cat_ranges, grand_total


def draw_panel_b(ax_main, ax_inset):
    flow = pd.read_csv(PANEL_C_DIR / "cds_reference_change_type_flow_grouped_counts.tsv", sep="\t")
    order = [
        "Exact match",
        "In-frame shorter",
        "In-frame longer",
        "Frameshift shorter",
        "Frameshift longer",
        "Coding lost",
        "Coding gained",
        "Non-coding",
    ]
    order = [
        lab for lab in order
        if (flow["ensembl_plot_group"].eq(lab).any() or flow["cat_plot_group"].eq(lab).any())
    ]
    rows_inset = flow[
        ~((flow["ensembl_plot_group"] == "Exact match") & (flow["cat_plot_group"] == "Exact match"))
    ].copy()
    main_colors = {
        "Exact match": COLORS["Exact match"],
        "In-frame shorter": "#b0b0b0",
        "In-frame longer": "#9a9a9a",
        "Frameshift shorter": "#7f7f7f",
        "Frameshift longer": "#666666",
        "Coding lost": "#8c8c8c",
        "Coding gained": "#737373",
        "Non-coding": "#c2c2c2",
    }

    def exact_label(lab, side):
        return lab == "Exact match"

    _, _, total = draw_cds_alluvial(
        ax_main,
        flow,
        order,
        main_colors,
        show_labels=True,
        label_filter=exact_label,
        label_fmt="{pct:.1f}%",
        label_bbox=False,
        fontsize=8,
        x_ens=0.18,
        x_cat=0.82,
        bar_w=0.08,
        label_offset=0.10,
        ribbon_alpha=0.15,
    )
    ax_main.set_xticklabels(["Ensembl", "CAT"], fontsize=8, fontweight="bold")
    ax_main.spines[["top", "right", "left"]].set_visible(False)

    draw_cds_alluvial(
        ax_inset,
        rows_inset,
        order,
        COLORS,
        percent_denominator=total,
        show_labels=True,
        label_fmt="{pct:.1f}%",
        min_label_pct=0.05,
        label_bbox=True,
        fontsize=7.5,
        x_ens=0.18,
        x_cat=0.82,
        bar_w=0.055,
        label_offset=0.045,
        ribbon_alpha=0.30,
    )
    ax_inset.tick_params(axis="x", pad=1, length=0)
    for spine in ax_inset.spines.values():
        spine.set_visible(True)
        spine.set_color("#666666")
        spine.set_linewidth(0.8)

    legend_keys = ["Exact match", "In-frame shorter", "In-frame longer", "Frameshift shorter", "Frameshift longer", "Coding lost"]
    legend_keys = [k for k in legend_keys if k in order]
    handles = [
        (
            patches.Patch(facecolor=main_colors.get(k, "#999999"), edgecolor="none"),
            patches.Patch(facecolor=COLORS[k], edgecolor="none"),
        )
        for k in legend_keys
    ]
    ax_inset.legend(
        handles=handles,
        labels=legend_keys,
        handler_map={tuple: HandlerTuple(ndivide=None, pad=0.20)},
        title="CDS change type (main / inset)",
        loc="upper center",
        bbox_to_anchor=(0.50, -0.32),
        ncol=2,
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        handlelength=1.1,
        columnspacing=0.8,
        labelspacing=0.25,
    )


def compact_count_label(x):
    if x >= 1000:
        if x < 10000 and x % 1000:
            return f"{x / 1000:.1f}k"
        return f"{x / 1000:.0f}k"
    return f"{x:.0f}"


def collapse_transcript_classes(df):
    mapping = {
        "Exact_Match": "Exact match",
        "Intron_Match": "Same intron chain",
        "Intron_Subset": "Same intron chain",
        "Intron_Superset": "Same intron chain",
        "Partial_5": "Partial overlap",
        "Partial_3": "Partial overlap",
        "Partial_Overlap": "Partial overlap",
        "Other_Partial": "Partial overlap",
        "No_Match": "No match",
        "Gene_Not_Shared": "Gene not shared",
    }
    out = df.copy()
    out["classification_group"] = out["classification"].map(mapping).fillna(out["classification"])
    return out


def source_to_median_counts(path):
    df = collapse_transcript_classes(pd.read_csv(path, sep="\t"))
    grouped = (
        df.groupby(["assembly_accession", "direction", "biotype", "classification_group"], as_index=False)["n_transcripts"].sum()
    )
    med = (
        grouped.groupby(["direction", "biotype", "classification_group"], as_index=False)["n_transcripts"].median()
        .rename(columns={"n_transcripts": "median_transcript_count"})
    )
    return med


def draw_panel_concordance_counts(fig, left, bottom, width, height, layout="pivot"):
    base = PANEL_A_DIR
    full_df = source_to_median_counts(base / "intron_chain_full_denom_per_assembly.tsv")
    matched_df = source_to_median_counts(base / "intron_chain_by_biotype_per_assembly.tsv")
    biotypes = ["protein_coding", "lncRNA", "pseudogene", "other_ncRNA", "other"]
    cats = ["Exact match", "Same intron chain", "Partial overlap", "No match", "Gene not shared"]
    gap = 0.018
    ax_w = (width - gap * (len(biotypes) - 1)) / len(biotypes)
    row_gap = 0.065
    ax_h = (height - row_gap) / 2
    axes = []
    for i, bt in enumerate(biotypes):
        if layout == "full_matched_rows":
            row_specs = [("Full feature set", full_df), ("Matched gene pairs", matched_df)]
        else:
            row_specs = [("Ensembl vs CAT", "Ensembl_to_CAT"), ("CAT vs Ensembl", "CAT_to_Ensembl")]

        for row, row_spec in enumerate(row_specs):
            ax_b = bottom + (1 - row) * (ax_h + row_gap)
            ax = fig.add_axes([left + i * (ax_w + gap), ax_b, ax_w, ax_h])
            axes.append(ax)
            max_total = 0
            if layout == "full_matched_rows":
                row_label, df = row_spec
                bar_specs = [(0, "Ensembl", "Ensembl_to_CAT"), (0.62, "CAT", "CAT_to_Ensembl")]
            else:
                row_label, direction = row_spec
                bar_specs = [(0, "Full", direction), (0.62, "Matched", direction)]
            for x, bar_label, direction in bar_specs:
                if layout == "pivot":
                    df = full_df if bar_label == "Full" else matched_df
                sub = df[(df["biotype"] == bt) & (df["direction"] == direction)]
                vals = [float(sub[sub["classification_group"] == c]["median_transcript_count"].iloc[0]) if (sub["classification_group"] == c).any() else 0 for c in cats]
                stack_base = 0
                for c, v in zip(cats, vals):
                    ax.bar(x, v, bottom=stack_base, width=0.48, color=COLORS[c], edgecolor="white", linewidth=0.35)
                    stack_base += v
                max_total = max(max_total, stack_base)
                exact_pct = 100 * vals[0] / sum(vals) if sum(vals) else 0
                ax.text(x, vals[0] / 2, f"{exact_pct:.0f}%", ha="center", va="center", fontsize=6.5, color="white", fontweight="bold")
                ax.text(x, stack_base * 1.025, compact_count_label(stack_base), ha="center", va="bottom", fontsize=6.5, color="#555555")
            if max_total:
                ax.set_ylim(0, max_total * 1.14)
            if row == 0:
                ax.set_title(BIOTYPE_LABELS[bt], fontsize=8, pad=2)
            ax.set_xlim(-0.34, 0.96)
            ax.set_xticks([0, 0.62])
            ax.set_xticklabels([bar_specs[0][1], bar_specs[1][1]], fontsize=6.5)
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: compact_count_label(x)))
            ax.tick_params(axis="y", labelsize=6.5)
            ax.grid(axis="y", color="#dddddd", linewidth=0.45)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)
            if i == 0:
                ax.set_ylabel(f"{row_label}\nmedian transcripts", fontsize=7)
    handles = [patches.Patch(facecolor=COLORS[c], label=c) for c in cats]
    axes[-1].legend(handles=handles, loc="lower left", bbox_to_anchor=(-4.95, -0.55), ncol=5, frameon=False, fontsize=7, handlelength=1.1, columnspacing=0.8)


def main(layout="pivot", suffix="layout_v8"):
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.6,
    })
    fig = plt.figure(figsize=(11.0, 8.2), facecolor="white")
    panel_letter(fig, 0.035, 0.965, "A")
    panel_letter(fig, 0.035, 0.460, "B")
    panel_letter(fig, 0.535, 0.460, "C")

    fig.text(0.075, 0.948, "Transcript concordance is high across matched feature sets", fontsize=10, ha="left", va="top")
    draw_panel_concordance_counts(fig, 0.075, 0.560, 0.860, 0.330, layout=layout)

    ax_b = fig.add_axes([0.095, 0.155, 0.390, 0.240])
    draw_panel_a(ax_b)

    ax_c_main = fig.add_axes([0.570, 0.142, 0.135, 0.270])
    ax_c_inset = fig.add_axes([0.735, 0.225, 0.230, 0.175])
    draw_panel_b(ax_c_main, ax_c_inset)

    pdf = FIGURE_DIR / f"comparative_annotation_supp_figure_{suffix}.pdf"
    png = FIGURE_DIR / f"comparative_annotation_supp_figure_{suffix}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(pdf)
    print(png)




In [ ]:
# Render both requested figure versions.
main(layout='pivot', suffix='with_pivot')
main(layout='full_matched_rows', suffix='full_top_matched_bottom')

print('Figure outputs:')
for path in sorted(FIGURE_DIR.glob('comparative_annotation_supp_figure_*')):
    print(' -', path.relative_to(OUTPUT_ROOT))


In [ ]:
# Write a compact manifest linking panels to source-data files and rendered figures.
manifest_rows = [
    {
        'panel': 'A',
        'description': 'Transcript concordance by feature class for full and matched annotation comparisons.',
        'source_data': str((PANEL_A_DIR / 'panel_A_transcript_concordance_source.tsv').relative_to(OUTPUT_ROOT)),
    },
    {
        'panel': 'A',
        'description': 'Full feature-set workflow summary used in Panel A.',
        'source_data': str((PANEL_A_DIR / 'intron_chain_full_denom_per_assembly.tsv').relative_to(OUTPUT_ROOT)),
    },
    {
        'panel': 'A',
        'description': 'Matched feature-set workflow summary used in Panel A.',
        'source_data': str((PANEL_A_DIR / 'intron_chain_by_biotype_per_assembly.tsv').relative_to(OUTPUT_ROOT)),
    },
    {
        'panel': 'B',
        'description': 'Within-biotype percentage breakdown of divergence categories relative to GENCODE v47.',
        'source_data': str((PANEL_B_DIR / 'divergence_by_biotype_panel_B_source.tsv').relative_to(OUTPUT_ROOT)),
    },
    {
        'panel': 'C',
        'description': 'Protein-coding CDS reference change type counts for the main bars and alluvial inset.',
        'source_data': str((PANEL_C_DIR / 'cds_reference_change_type_flow_grouped_counts.tsv').relative_to(OUTPUT_ROOT)),
    },
    {
        'panel': 'all',
        'description': 'Provenance table recording which input file produced each source-data output.',
        'source_data': str((SOURCE_DATA_DIR / 'source_data_provenance.tsv').relative_to(OUTPUT_ROOT)),
    },
]

for fig_path in sorted(FIGURE_DIR.glob('comparative_annotation_supp_figure_*.pdf')):
    manifest_rows.append({
        'panel': 'figure',
        'description': fig_path.stem,
        'source_data': str(fig_path.relative_to(OUTPUT_ROOT)),
    })

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_ROOT / 'source_data_manifest.tsv'
manifest.to_csv(manifest_path, sep='	', index=False)
print(manifest_path)
manifest


## Notes for final handover

- For the final HPC rerun, set `FULL_FEATURE_RUN_OUTDIR` and `MATCHED_FEATURE_RUN_OUTDIR` to the two completed Nextflow output directories.
- `DIVERGENCE_RUN_OUTDIR` should usually be the matched/filtered run, because Panels B and C describe differences among the matched annotations.
- Set `ALLOW_PRECOMPUTED_FALLBACK = False` for the final check if you want the notebook to fail whenever Panel B/C cannot be generated from workflow summaries.
- This notebook intentionally does not build the supplementary tables. Keep using `supplementary_tables.ipynb`, with the final Supplementary Table 3 filtered to `n_cds_reference_discordant_assemblies >= 2`.
- The gene-name missing table should be generated separately from GENCODE v47 gene names versus each cached annotation GFF.
